In [4]:
import pandas as pd
import numpy as np
from pathlib import Path

# مسار البيانات المعالجة
PROCESSED_DIR = Path("../data/processed/part_0001.parquet")

# 1. قراءة البيانات (Parquet يقرأ المجلد كاملاً كأنه ملف واحد)
print("📂 Loading Processed Data...")
try:
    df = pd.read_parquet(PROCESSED_DIR)
    print(f"✅ Loaded successfully! Shape: {df.shape}")
except Exception as e:
    print(f"❌ Error loading data: {e}")
    # توقف هنا إذا فشل التحميل

# 2. عرض عينة عشوائية
print("\n🔍 Random Sample:")
display(df)

# 3. فحص أنواع البيانات (Data Types)
print("\n📊 Data Types:")
print(df.dtypes)

# 4. فحص عمود القوائم (Diagnoses List) - أهم اختبار!
# نتأكد أنها List حقيقية وليست نصاً
print("\n🧪 Inspecting Diagnoses Lists:")
sample_diag = df['diagnosis_names'].iloc[0]
print(f"Type of cell content: {type(sample_diag)}")
print(f"Content example: {sample_diag}")

if isinstance(sample_diag, np.ndarray) or isinstance(sample_diag, list):
    print("✅ GREAT! Diagnoses are stored as Arrays/Lists.")
else:
    print("⚠️ WARNING: Diagnoses might be stored as Strings!")

# 5. فحص القيم المفقودة (Nulls)
print("\n🕳️ Missing Values Check:")
print(df.isnull().sum())

# 6. فحص المنطق (Business Logic Check)
# هل هناك مريض لديه وصفة دواء ولكن لا يوجد له عمر أو وزن؟ (مشكلة في الـ Join)
orphan_records = df[df['anchor_age'].isnull()]

# 7. فحص التزامن بين الأكواد والأسماء (Alignment Check)
print("\n⚖️ Checking Alignment (Codes vs Names lengths)...")

# دالة لحساب الفرق في الطول
def check_length_mismatch(row):
    return len(row['diagnosis_codes']) != len(row['diagnosis_names'])

# تطبيق الفحص
mismatches = df[df.apply(check_length_mismatch, axis=1)]

if len(mismatches) > 0:
    print(f"⚠️ CRITICAL WARNING: Found {len(mismatches)} rows where Codes count != Names count.")
    display(mismatches[['hadm_id', 'diagnosis_codes', 'diagnosis_names']].head(3))
else:
    print("✅ PERFECT ALIGNMENT: Every diagnosis code has a corresponding name.")
    
if len(orphan_records) > 0:
    print(f"\n⚠️ Alert: Found {len(orphan_records)} prescriptions with missing patient info (Join failed for some IDs).")
else:
    print("\n✅ Integrity Check Passed: All prescriptions link to a patient.")

📂 Loading Processed Data...
❌ Error loading data: [Errno 2] No such file or directory: '..\\data\\processed\\part_0001.parquet'

🔍 Random Sample:


,subject_id,hadm_id,starttime,drug,ndc,dose_val_rx,dose_unit_rx,gender,anchor_age,weight,diagnosis_codes,diagnosis_names
0,10000032,22595853,2180-05-08 08:00:00,Furosemide,5.107901e+10,40,mg,F,52,93.104,"[5723, 78959, 5715, 07070, 496, 29680, 30981, ...","[Portal hypertension, Other ascites, Cirrhosis..."
1,10000032,22595853,2180-05-07 02:00:00,Ipratropium Bromide Neb,4.879801e+08,1,NEB,F,52,93.104,"[5723, 78959, 5715, 07070, 496, 29680, 30981, ...","[Portal hypertension, Other ascites, Cirrhosis..."
2,10000032,22595853,2180-05-07 01:00:00,Furosemide,5.107901e+10,20,mg,F,52,93.104,"[5723, 78959, 5715, 07070, 496, 29680, 30981, ...","[Portal hypertension, Other ascites, Cirrhosis..."
3,10000032,22595853,2180-05-07 01:00:00,Potassium Chloride,2.450041e+08,40,mEq,F,52,93.104,"[5723, 78959, 5715, 07070, 496, 29680, 30981, ...","[Portal hypertension, Other ascites, Cirrhosis..."
4,10000032,22595853,2180-05-07 00:00:00,Sodium Chloride 0.9% Flush,0.000000e+00,3,mL,F,52,93.104,"[5723, 78959, 5715, 07070, 496, 29680, 30981, ...","[Portal hypertension, Other ascites, Cirrhosis..."
...,...,...,...,...,...,...,...,...,...,...,...,...
99995,10049746,24332085,2136-11-26 12:00:00,CycloPHOSPHAMIDE,1.001910e+10,680,mg,F,72,144.770,"[C8339, I2699, E43, J189, S72302A, E872, E870,...","[Diffuse large B-cell lymphoma, extranodal and..."
99996,10049746,24332085,2137-01-01 08:00:00,Ondansetron,5.107905e+10,16,mg,F,72,144.770,"[C8339, I2699, E43, J189, S72302A, E872, E870,...","[Diffuse large B-cell lymphoma, extranodal and..."
99997,10049746,24332085,2136-11-26 20:00:00,OLANZapine (Disintegrating Tablet),5.026806e+10,2.5,mg,F,72,144.770,"[C8339, I2699, E43, J189, S72302A, E872, E870,...","[Diffuse large B-cell lymphoma, extranodal and..."
99998,10049746,24332085,2136-11-27 21:00:00,D5W,0.000000e+00,300,mL,F,72,144.770,"[C8339, I2699, E43, J189, S72302A, E872, E870,...","[Diffuse large B-cell lymphoma, extranodal and..."



📊 Data Types:
subject_id                  Int64
hadm_id                     Int64
starttime          datetime64[ns]
drug                       object
ndc                       float64
dose_val_rx                object
dose_unit_rx               object
gender                   category
anchor_age                  int64
weight                    float64
diagnosis_codes            object
diagnosis_names            object
dtype: object

🧪 Inspecting Diagnoses Lists:
Type of cell content: <class 'numpy.ndarray'>
Content example: ['Portal hypertension' 'Other ascites'
 'Cirrhosis of liver without mention of alcohol'
 'Unspecified viral hepatitis C without hepatic coma'
 'Chronic airway obstruction, not elsewhere classified'
 'Bipolar disorder, unspecified' 'Posttraumatic stress disorder'
 'Personal history of tobacco use']
✅ GREAT! Diagnoses are stored as Arrays/Lists.

🕳️ Missing Values Check:
subject_id             0
hadm_id                0
starttime            102
drug                  

In [8]:
import pandas as pd
import numpy as np
from pathlib import Path

# 1. تحديد مسار الملف الذهبي الجديد
GOLD_FILE = Path("../data/processed/GOLD_patient_records.parquet")

print(f"📂 Loading GOLD Dataset from: {GOLD_FILE.name} ...")

try:
    df = pd.read_parquet(GOLD_FILE)
    print(f"✅ Loaded successfully! Shape: {df.shape}")
    print(f"   (Expectation: Rows = Number of unique patients)")
except Exception as e:
    print(f"❌ Error loading data: {e}")
    # توقف هنا
    raise e

# 2. عرض عينة لفهم الهيكلية الجديدة
print("\n🔍 Random Patient Profile:")
# نوسع العرض لنرى القوائم كاملة
pd.set_option('display.max_colwidth', 100) 
display(df.sample(10))

# 3. اختبار "مريض واحد = صف واحد" (Uniqueness Check)
print("\n🆔 Checking Patient Uniqueness...")
if df['subject_id'].is_unique:
    print(f"✅ PASSED: All {len(df)} rows represent unique patients.")
else:
    print(f"⚠️ FAILED: Duplicates found! ({df['subject_id'].duplicated().sum()} duplicates)")

# 4. فحص القوائم (Diagnosis Names & Medication List)
print("\n🧪 Inspecting Lists Structure:")

for col in ['diagnosis_names', 'medication_list']:
    print(f"   Checking column: {col}...")
    if col not in df.columns:
        print(f"   ❌ Missing column {col}!")
        continue
        
    sample_val = df[col].iloc[0]
    print(f"      -> Type: {type(sample_val)}")
    print(f"      -> Sample Content: {sample_val[:10]} ... (Truncated)")
    
    if isinstance(sample_val, (list, np.ndarray)):
        # فحص هل القائمة فارغة؟
        empty_count = df[col].apply(lambda x: len(x) == 0).sum()
        print(f"      ✅ Valid List Format. (Empty lists count: {empty_count})")
    else:
        print("      ⚠️ WARNING: Not a list!")

# 5. فحص القيم المفقودة (Nulls)
print("\n🕳️ Missing Values Check:")
print(df.isnull().sum())

# 6. فحص منطقي (هل يوجد دواء بدون تشخيص؟)
# هذا ليس خطأ بالضرورة، لكنه مفيد للمودل
print("\n🧠 Logic Check:")
no_meds = df[df['medication_list'].apply(len) == 0]
no_diag = df[df['diagnosis_names'].apply(len) == 0]

print(f"   - Patients with NO medications: {len(no_meds)}")
print(f"   - Patients with NO diagnoses: {len(no_diag)}")

if len(no_meds) == 0 and len(no_diag) == 0:
    print("✅ Excellent! Every patient has at least one med and one diagnosis.")

📂 Loading GOLD Dataset from: GOLD_patient_records.parquet ...
✅ Loaded successfully! Shape: (196738, 9)
   (Expectation: Rows = Number of unique patients)

🔍 Random Patient Profile:


,subject_id,hadm_id,gender,anchor_age,weight,medication_list,dosage_list,unit_list,diagnosis_names
130517,16659685,23030610,M,23,NaN,"[0.45% Sodium Chloride, 0.9% Sodium Chloride, Simvastatin, NS, MetRONIDAZOLE (FLagyl), 0.45% Sod...","[1000, 1000, 20, 100, 500, 1000, 100, 4, 500, 200, 400, 3, 1000, 4, 0.5]","[mL, mL, mg, mL, mg, mL, mL, gm, mg, mL, mg, mL, mL, mg, mL]","[Syncope and collapse, Thrombocytopenia, unspecified, Intestinal infection due to other organism..."
97306,14972476,28430535,F,59,167.144444,"[Pantoprazole, Clonazepam, Heparin, Clonazepam, Sodium Chloride 0.9% Flush, Levothyroxine Sodiu...","[40, 1, 5000, 1, 3, 112, 600, 250, 40, 1000, 1-2, 0.25, 15, 100, 20, 500, 30, 2]","[mg, mg, UNIT, mg, mL, mcg, mg, mg, mg, mL, TAB, mg, mg, mg, mg, mg, mg, mg]","[Benign neoplasm of ovary, Polyp of corpus uteri, Unspecified acquired hypothyroidism, Unspecifi..."
139342,17106592,27564537,M,66,146.025000,"[Acetaminophen, OxyCODONE (Immediate Release), Lactated Ringers, Ciprofloxacin HCl, Influenza Va...","[650, 5, 1000, 500, 0.5, 3-10, 50, 400, 1, 2, 200, 1000, 100, 500, 1000, 8.6, 1000, 1, 3, 40, 10...","[mg, mg, mL, mg, mL, mL, mg, mg, VIAL, g, mL, mg, mL, mg, mL, mg, mL, VIAL, g, mg, mg, mg, g, mL...","[Abscess of liver, Klebsiella pneumoniae [K. pneumoniae] as the cause of diseases classified els..."
37956,11943894,29328542,M,59,329.250000,"[Indomethacin, Diazepam, Aluminum-Magnesium Hydrox.-Simethicone, Gabapentin, Ondansetron ODT, Di...","[50, 5, 30, 300, 4, 10, 10, 6, 2, 10, 30, 40, 50, 300, 100, 1, 600, 650, 5, 50, 1]","[mg, mg, mL, mg, mg, mg, mg, mg, mg, mg, mL, mg, mg, mg, mg, Appl, mg, mg, mg, mg, mg]","[Depressive disorder, not elsewhere classified, Cocaine dependence, continuous, Chronic pancreat..."
186700,19487571,20562493,M,76,202.000000,"[Senna, Potassium Chloride, Losartan Potassium, Warfarin, Atorvastatin, Eplerenone, Warfarin, PN...","[8.6, 40, 50, 3, 80, 25, 1, 0.5, 1, 40, 325-650, 60, 0.125, 1000, 100, 0.5, 125, 0.4, 60, 5000, ...","[mg, mEq, mg, mg, mg, mg, dose, mL, mg, mg, mg, mEq, mg, mL, mg, mL, mcg, mg, mEq, UNIT, mg, mg,...","[Ventricular tachycardia, Chronic systolic (congestive) heart failure, Unspecified atrial flutte..."
49027,12507121,22610205,F,43,NaN,"[Potassium Chl 20 mEq / 1000 mL D5 1/2 NS, Sodium Chloride 0.9% Flush, DiphenhydrAMINE, HYDROmo...","[1000, 3, 50, 0.25, 1000, 0.5, 200, 400, 0.5, 100, 500, 1000, 20, 4, 5000]","[mL, mL, mg, mg, mg, mL, mL, mg, mL, mL, mg, mL, mg, mg, UNIT]","[Urinary tract infection, site not specified, Esophageal reflux, Asthma, unspecified type, unspe..."
135283,16901518,28918394,M,82,NaN,"[5% Dextrose, Heparin Sodium, Cholestyramine, Clopidogrel, Albuterol-Ipratropium, Morphine Sulfa...","[250, 25,000, 4, 75, 1-2, 2-4, 0.5, 3, 2-4, 325, 12.5, 6.25, 12.5, 1, 1, 75, 10, 20, 325, 6.25, ...","[mL, UNIT, gm, mg, PUFF, mg, mL, mL, mg, mg, mg, mg, gm, CAP, INH, mg, mg, mg, mg, mg, mg, mL, m...","[Subendocardial infarction, initial episode of care, Unspecified essential hypertension, Pure hy..."
116436,15944096,29867421,F,69,194.540000,"[LOPERamide, Sodium Chloride 0.9% Flush, Enoxaparin Sodium, Sertraline, Levothyroxine Sodium, G...","[2, 3, 30, 100, 50, 1, 1, 50-100, 8.6, 1000, 40, 4, 5-10, 1000, 150, 0, 1000, 50, 2, 100, 1000, ...","[mg, mL, mg, mg, mcg, mg, mg, mg, mg, mL, mg, mg, mg, mL, mg, UNIT, mg, mL, g, mg, mg, mg, mg, m...","[Unilateral primary osteoarthritis, right knee, Cardiomyopathy, unspecified, Unspecified atrial ..."
59972,13061918,22528033,F,26,141.700000,"[Acetaminophen, Metoprolol Tartrate, amLODIPine, Bag, Magnesium Sulfate, Bag, Magnesium Sulfate,...","[650, 6.25, 5, 1, 2, 1, 2, 1500, 5, 1000, 1000, 3-10, 1000, 5, 1000, 6.25, 2.5, 1000, 40, 40, 40...","[mg, mg, mg, BAG, gm, BAG, gm, mg, mg, mL, mg, mL, mL, mg, mL, mg, mg, mg, mg, mg, mEq, mL, mg, ...","[Viral meningitis, unspecified, Rhabdomyolysis, Takotsubo syndrome, Other reaction to spinal and..."
189128,19610569,21883209,M,33,206.200000,"[Potassium Chloride, Spironolactone, Prochlorp


🆔 Checking Patient Uniqueness...
✅ PASSED: All 196738 rows represent unique patients.

🧪 Inspecting Lists Structure:
   Checking column: diagnosis_names...
      -> Type: <class 'numpy.ndarray'>
      -> Sample Content: ['Chronic hepatitis C without mention of hepatic coma' 'Other ascites'
 'Other dependence on machines, supplemental oxygen'
 'Cirrhosis of liver without mention of alcohol' 'Hyperpotassemia'
 'Hyposmolality and/or hyponatremia'
 'Chronic airway obstruction, not elsewhere classified'
 'Asymptomatic human immunodeficiency virus [HIV] infection status'
 'Tobacco use disorder' 'Diarrhea'] ... (Truncated)
      ✅ Valid List Format. (Empty lists count: 134)
   Checking column: medication_list...
      -> Type: <class 'numpy.ndarray'>
      -> Sample Content: ['Sodium Chloride 0.9%  Flush' '0.9% Sodium Chloride' 'Calcium Gluconate'
 'Furosemide' 'Albuterol Inhaler' 'Sodium Polystyrene Sulfonate'
 'Zolpidem Tartrate' 'Emtricitabine-Tenofovir (Truvada)' 'Rifaximin'
 'Sodium Pol